In [ ]:
from collections.abc import Mapping
from pathlib import Path
from typing import Any
import platform
import sys
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
WORKFLOW = 'metabolomics'  # or 'lipidomics'
METAB_DATA_DIR = Path.cwd().parent / '00-metabolomics_tardis'
LIPID_DATA_DIR = Path.cwd().parent / '00-lipidomics_tardis'
METAB_TARGET_LIST = Path('TARDIS_Target List_POLAR Metabolomics.xlsx')
LIPID_TARGET_LIST = Path('TARDIS_Target List_LIPIDOMICS.xlsx')
COMPONENT_QUALITY_FILE = Path('component_quality.xlsx')

In [ ]:
def fillna_random_uniform(df: pd.DataFrame):
    """
    Fill missing values in each column with random values drawn from a uniform
    distribution between half the column's minimum and the minimum itself.

    Based on:
    https://rformassspectrometry.github.io/Metabonaut/articles/a-end-to-end-untargeted-metabolomics.html#initial-quality-assessment
    """
    return df.pipe(
        lambda df: df.assign(
            **{
                col: df[col].fillna(
                    pd.Series(
                        np.random.uniform(
                            df[col].min() / 2,
                            df[col].min(),
                            size=df[col].isna().sum(),
                        ),
                        index=df[df[col].isna()].index,
                    )
                )
                for col in df.columns
            }
        )
    )

def prepare_and_export_dataset(
    df: pd.DataFrame,
    target_list: Mapping[Any, str],
    component_quality_dict: Mapping[Any, str],
    output_file: str | Path,
) -> pd.DataFrame:
    processed_df = (
        df
        .groupby(df.index)
        .max()
        .T
        .pipe(lambda df: fillna_random_uniform(df))
        .assign(name=lambda df: df.index.map(target_list))
        .rename_axis('id')
        .assign(peak_quality=lambda df: df.index.map(component_quality_dict))
    )
    processed_df.to_csv(output_file)
    return processed_df

In [ ]:
component_quality_dict = (pd.read_excel(COMPONENT_QUALITY_FILE).assign(to_keep=lambda df: df.to_keep.map({0: 'Bad', 1: 'Good'})).set_index('id').to_keep.to_dict())

if WORKFLOW == 'metabolomics':
    source_files = sorted(METAB_DATA_DIR.glob('*.csv'))
    target_list = pd.read_excel(METAB_TARGET_LIST).set_index('id')['name'].to_dict()
    output_file = OUTPUT_DIR / 'metabolomics_combined_gapfilled.csv'
elif WORKFLOW == 'lipidomics':
    source_files = sorted(LIPID_DATA_DIR.glob('*.csv'))
    target_list = pd.read_excel(LIPID_TARGET_LIST).set_index('id')['name'].to_dict()
    output_file = OUTPUT_DIR / 'lipidomics_combined_gapfilled.csv'
else:
    raise ValueError("WORKFLOW must be 'metabolomics' or 'lipidomics'.")

if not source_files:
    raise FileNotFoundError('No input CSV files found.')
raw_data = pd.concat([pd.read_csv(path, index_col=1).drop(columns='Unnamed: 0') for path in source_files], axis=1).T
processed_data = prepare_and_export_dataset(raw_data, target_list, component_quality_dict, output_file)
print(f'Wrote {processed_data.shape[0]} samples to {output_file}')

record = OUTPUT_DIR / f'{WORKFLOW}_gapfill_environment.txt'
record.write_text('\n'.join([f'Random seed: {RANDOM_SEED}', f'Python: {sys.version}', f'Platform: {platform.platform()}', f'numpy: {np.__version__}', f'pandas: {pd.__version__}]) + '\n')